[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_03/ATS_ch3_lp_vs_var/ATS_ch3_lp_vs_var.ipynb)

# ATS_ch3_lp_vs_var

Local projections against VARs: a simulation of the bias-variance trade-off (Li, Plagborg-Moller and Wolf 2024) with a misspecified VAR(2), a VAR(12) and LP(2); LP-IV with the Gertler-Karadi instrument (Ramey 2016, Figure 3B specification) against the proxy SVAR.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 3: Structural VAR and local projections. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_3'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io

import zipfile

import urllib.error

SEED = 2026

RAMEY_MON = 'https://econweb.ucsd.edu/~vramey/research/Ramey_HOM_monetary.zip'

RAMEY_RZ = 'https://econweb.ucsd.edu/~vramey/research/Ramey_Zubairy_replication_codes.zip'

EIA_INTL = 'https://api.eia.gov/bulk/INTL.zip'

EIA_PROD = 'INTL.57-1-WORL-TBPD.M'

EIA_MER9 = 'https://www.eia.gov/totalenergy/data/browser/csv.php?tbl=T09.01'

EA_Y1 = ('YC', 'B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y')

JK_ECB = 'https://raw.githubusercontent.com/marekjarocinski/jkshocks_update_ecb/main/shocks_ecb_mpd_me_m.csv'

RO_IP = ('sts_inpr_m', 'M.PRD.B-D.SCA.I21.RO')

EA_IP = ('sts_inpr_m', 'M.PRD.B-D.SCA.I21.EA20')

RO_HICP = ('prc_hicp_minr', 'M.I25.TOTAL.RO')

EA_HICP = ('prc_hicp_minr', 'M.I25.TOTAL.EA')

RO_R3M = ('irt_st_m', 'M.IRT_M3.RO')

EA_R3M = ('irt_st_m', 'M.IRT_M3.EA')

RO_START = '2005-08-01'

CEE = {'vars': ['lip', 'unemp', 'lcpi', 'lpcom', 'ffr', 'lnbr', 'ltr', 'lm1'], 'p': 12, 'start': '1965-01-01', 'end': '1995-06-01', 'H': 48, 'shock': 'ffr'}

KIL = {'p': 24, 'start': '1973-01-01', 'end': '2007-12-01', 'H': 18}

BQ = {'p': 8, 'start': '1948-04-01', 'end': '1987-10-01', 'brk': '1974-01-01', 'H': 40}

UHL = {'vars': ['lip', 'lcpi', 'lpcom', 'ffr', 'lnbr', 'ltr'], 'p': 12, 'start': '1965-01-01', 'end': '2003-12-01', 'K': 5, 'H': 60}

GK = {'vars': ['gs1', 'lip', 'lcpi', 'ebp'], 'p': 12, 'start': '1979-07-01', 'end': '2012-06-01', 'zstart': '1991-01-01', 'H': 48}

LPGK = {'p': 2, 'start': '1990-01-01', 'end': '2012-06-01', 'H': 48}

RZ = {'p': 4, 'H': 20, 'thr': 6.5, 'start': 1889.0}

RO = {'p': 2, 'H': 36, 'end': '2026-07-01'}

NBOOT = 500

RO_VARS = ['ip_ea', 'p_ea', 'r_ea', 'ip_ro', 'p_ro', 'r_ro', 'fx']

PSI = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 0.8, 0.6, 0.4, 0.3, 0.2, 0.1])

KIL_SIGN = np.diag([-1.0, 1.0, 1.0])

_FILES = {}

def read_ecb(flow, key, start=None):
    """One series from the ECB Data Portal (public SDMX REST API, CSV), e.g. the monthly USD/EUR reference rate:
        read_ecb('EXR', 'M.USD.EUR.SP00.A')
    or the euro-area HICP annual rate of change: read_ecb('ICP', 'M.U2.N.000000.4.ANR')."""
    url = (f'https://data-api.ecb.europa.eu/service/data/{flow}/{key}?format=csvdata'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('ecb', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{flow}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def get_bytes(url):
    """Download a public file once per session (no key); optional local cache folder ATS_CACHE."""
    import hashlib
    import time
    if url in _FILES:
        return _FILES[url]
    cache = os.environ.get('ATS_CACHE')
    path = os.path.join(cache, hashlib.md5(url.encode()).hexdigest()) if cache else None
    if path and os.path.exists(path):
        _FILES[url] = open(path, 'rb').read()
        return _FILES[url]
    for attempt in range(5):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (ATS course)'})
            _FILES[url] = urllib.request.urlopen(req, timeout=300).read()
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 4:
                raise
            time.sleep(10 * (attempt + 1))
    if path:
        os.makedirs(cache, exist_ok=True)
        open(path, 'wb').write(_FILES[url])
    return _FILES[url]


def zip_member(url, name):
    """One file from a public zip archive."""
    z = zipfile.ZipFile(io.BytesIO(get_bytes(url)))
    m = [f for f in z.namelist() if f.endswith(name)][0]
    return z.read(m)


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def lagmat(Y, p):
    """Regressor matrix [Y_{t-1}, ..., Y_{t-p}] for t = p..T-1 (rows), and the left-hand side Y_t."""
    Y = np.asarray(Y, float)
    T = len(Y)
    X = np.hstack([Y[p - j - 1:T - j - 1] for j in range(p)])
    return Y[p:], X


def var_ols(Y, p, const=True, exog=None):
    """OLS of a VAR(p) with optional constant and exogenous regressors (aligned with Y). Returns a dict with the lag
    matrices A (p x n x n), the deterministic coefficients, residuals U (T-p x n) and Sigma = U'U/(T - p - k)."""
    Yt, X = lagmat(Y, p)
    n = Yt.shape[1]
    D = []
    if const:
        D.append(np.ones((len(Yt), 1)))
    if exog is not None:
        D.append(np.asarray(exog, float)[p:])
    Xf = np.hstack([X] + D) if D else X
    B = np.linalg.lstsq(Xf, Yt, rcond=None)[0]
    U = Yt - Xf @ B
    k = Xf.shape[1]
    A = np.stack([B[j * n:(j + 1) * n].T for j in range(p)])
    return dict(A=A, det=B[n * p:], U=U, Sigma=U.T @ U / (len(Yt) - k), Sigma_ml=U.T @ U / len(Yt), X=Xf, B=B,
                Y=np.asarray(Y, float), p=p, const=const, exog=exog)


def companion(A):
    p, n, _ = A.shape
    F = np.zeros((n * p, n * p))
    F[:n] = np.hstack(list(A))
    F[n:, :-n] = np.eye(n * (p - 1))
    return F


def max_root(A):
    return float(np.max(np.abs(np.linalg.eigvals(companion(A)))))


def ma_coefs(A, H):
    """Reduced-form MA coefficients Phi_0..Phi_H (Phi_0 = I)."""
    p, n, _ = A.shape
    Phi = np.zeros((H + 1, n, n))
    Phi[0] = np.eye(n)
    for h in range(1, H + 1):
        Phi[h] = sum(A[j] @ Phi[h - j - 1] for j in range(min(p, h)))
    return Phi


def irf(A, B0, H):
    """Structural impulse responses Theta_h = Phi_h B0 (H+1 x n x n): [h, response, shock]."""
    return np.einsum('hij,jk->hik', ma_coefs(A, H), B0)


def fevd(Theta):
    """Forecast error variance decomposition: share of shock k in the h-step error variance of variable i."""
    c = np.cumsum(Theta ** 2, axis=0)
    return c / c.sum(axis=2, keepdims=True)


def simulate_var(A, det, Xdet, U, Y0):
    """Recursive simulation of a VAR from initial values Y0 (p x n) with residuals U and deterministic terms."""
    p, n, _ = A.shape
    T = len(U)
    Y = np.zeros((T + p, n))
    Y[:p] = Y0
    for t in range(T):
        y = sum(A[j] @ Y[p + t - j - 1] for j in range(p)) + U[t]
        if Xdet is not None:
            y = y + Xdet[t] @ det
        Y[p + t] = y
    return Y


def bootstrap_var(m, ident, H, B=NBOOT, kind='residual', seed=SEED, bias=None):
    """Bootstrap distribution of structural responses. ident(A, Sigma, U) -> B0. kind: 'residual' (i.i.d. resampling
    of residuals) or 'wild' (recursive-design wild bootstrap, Rademacher weights; Goncalves and Kilian 2004).
    bias: optional bias of A (Kilian 1998) subtracted from each bootstrap estimate."""
    rng = np.random.default_rng(seed)
    A, U, p, Y = m['A'], m['U'], m['p'], m['Y']
    n = A.shape[1]
    Xdet = m['X'][:, n * p:] if m['X'].shape[1] > n * p else None
    det = m['det'] if Xdet is not None else None
    Uc = U - U.mean(axis=0)
    out = np.zeros((B, H + 1, n, n))
    Abar = np.zeros_like(A)
    for b in range(B):
        Ub = Uc[rng.integers(0, len(U), len(U))] if kind == 'residual' else U * rng.choice([-1.0, 1.0], (len(U), 1))
        s = rng.integers(0, len(Y) - p + 1)
        Yb = simulate_var(A, det, Xdet, Ub, Y[s:s + p])
        mb = var_ols(Yb, p, const=m['const'], exog=None if m['exog'] is None else m['exog'])
        Ab = mb['A']
        Abar += Ab / B
        if bias is not None:
            Ab = shrink_bias(Ab, bias)
        out[b] = irf(Ab, ident(Ab, mb['Sigma'], mb['U']), H)
    return out, Abar


def shrink_bias(A, bias):
    """Kilian (1998): subtract the bias, shrinking the correction until the corrected VAR is stationary."""
    d = 1.0
    while d > 0:
        Ac = A - d * bias
        if max_root(Ac) < 1:
            return Ac
        d -= 0.01
    return A


def kilian_bias(m, B=200, seed=SEED + 1):
    """Bootstrap estimate of the small-sample bias of the VAR slope coefficients (Kilian 1998, first stage)."""
    _, Abar = bootstrap_var(m, lambda A, S, U: np.eye(A.shape[1]), 0, B=B, seed=seed)
    return Abar - m['A']


def bands(draws, lev=(0.05, 0.95)):
    return np.quantile(draws, lev[0], axis=0), np.quantile(draws, lev[1], axis=0)


def chol(A, S, U):
    return np.linalg.cholesky(S)


def band_plot(ax, h, mid, lo, hi, color, label, lo2=None, hi2=None):
    """Point estimate with a shaded band (and an optional inner band)."""
    ax.fill_between(h, lo, hi, color=color, alpha=0.15, lw=0, label='_band')
    if lo2 is not None:
        ax.fill_between(h, lo2, hi2, color=color, alpha=0.28, lw=0, label='_band2')
    ax.plot(h, mid, color=color, lw=1.8, label=label)
    ax.axhline(0, color=st.DarkText, lw=0.6)


def patch(color, alpha=0.2):
    from matplotlib.patches import Patch
    return Patch(color=color, alpha=alpha)


def nw_se(X, e, L):
    """Newey-West (Bartlett) covariance of OLS coefficients with L lags."""
    XtXi = np.linalg.inv(X.T @ X)
    g = X * e[:, None]
    S = g.T @ g
    for l in range(1, L + 1):
        w = 1 - l / (L + 1)
        G = g[l:].T @ g[:-l]
        S += w * (G + G.T)
    return XtXi @ S @ XtXi


def lp(y, x, W, H, nw=True, lag_aug=False):
    """Local projections (Jorda 2005): y_{t+h} on x_t and controls W_t (rows aligned with t), h = 0..H.
    Standard errors: Newey-West with h + 1 lags, or Eicker-Huber-White when lag_aug (Montiel Olea and
    Plagborg-Moller 2021: the controls include one extra lag)."""
    y, x, W = np.asarray(y, float), np.asarray(x, float), np.asarray(W, float)
    b, se = np.full(H + 1, np.nan), np.full(H + 1, np.nan)
    for h in range(H + 1):
        yy = y[h:]
        X = np.column_stack([x[:len(x) - h], np.ones(len(x) - h), W[:len(W) - h]])
        ok = ~np.isnan(yy) & ~np.isnan(X).any(axis=1)
        Xo, yo = X[ok], yy[ok]
        bh = np.linalg.lstsq(Xo, yo, rcond=None)[0]
        e = yo - Xo @ bh
        V = nw_se(Xo, e, 0 if lag_aug else h + 1)
        b[h], se[h] = bh[0], np.sqrt(V[0, 0])
    return b, se


def tsls(y, x, z, W):
    """Two-stage least squares of y on x (instrumented by z) with exogenous controls W; returns beta, residuals, design."""
    Z = np.column_stack([z, np.ones(len(z)), W])
    X = np.column_stack([x, np.ones(len(x)), W])
    Pz = Z @ np.linalg.lstsq(Z, X, rcond=None)[0]
    b = np.linalg.lstsq(Pz, y, rcond=None)[0]
    return b, y - X @ b, Pz


def lp_iv(y, x, z, W, H, cum=False):
    """LP-IV (Stock and Watson 2018): y_{t+h} on x_t instrumented by z_t, controls W_t; Newey-West SE (h + 1 lags).
    cum=True: cumulative y_{t..t+h} on cumulative x_{t..t+h} (the multiplier of Ramey and Zubairy 2018)."""
    y, x, z, W = (np.asarray(a, float) for a in (y, x, z, W))
    b, se, F = np.full(H + 1, np.nan), np.full(H + 1, np.nan), np.full(H + 1, np.nan)
    T = len(y)
    for h in range(H + 1):
        if cum:
            yy = np.array([y[t:t + h + 1].sum() if t + h < T else np.nan for t in range(T)])
            xx = np.array([x[t:t + h + 1].sum() if t + h < T else np.nan for t in range(T)])
        else:
            yy = np.r_[y[h:], np.full(h, np.nan)]
            xx = x
        D = np.column_stack([yy, xx, z, W])
        ok = ~np.isnan(D).any(axis=1)
        bh, e, Pz = tsls(yy[ok], xx[ok], z[ok], W[ok])
        V = nw_se(Pz, e, h + 1)
        b[h], se[h] = bh[0], np.sqrt(V[0, 0])
        Xf = np.column_stack([z[ok], np.ones(ok.sum()), W[ok]])
        g = np.linalg.lstsq(Xf, xx[ok], rcond=None)[0]
        ef = xx[ok] - Xf @ g
        Vf = nw_se(Xf, ef, h + 1)
        F[h] = g[0] ** 2 / Vf[0, 0]
    return b, se, F


def lags_of(df, cols, p, start=1):
    """Lags start..p of the columns `cols` of a DataFrame, as one array."""
    return np.column_stack([df[c].shift(j).values for c in cols for j in range(start, p + 1)])


def ramey_monthly():
    """Monthly US data of Ramey (2016), sheet 'Monthly' of Monetarydat.xlsx: log IP, unemployment, log CPI, log
    commodity prices, funds rate, log nonborrowed and total reserves, log M1, 1-year Treasury yield, excess bond
    premium, Gertler-Karadi FF4 surprises (ff4_tc), Romer-Romer shocks; 1959:1 onward."""
    d = pd.read_excel(io.BytesIO(zip_member(RAMEY_MON, 'Monetarydat.xlsx')), sheet_name='Monthly')
    d.columns = [c.lower() for c in d.columns]
    d.index = pd.date_range('1959-01-01', periods=len(d), freq='MS')
    return d


def first_stage(U, z):
    """Regression of the policy residual u1 on the instrument z: coefficient, robust t and F = t^2."""
    z = z - z.mean()
    u = U[:, 0] - U[:, 0].mean()
    b = (z @ u) / (z @ z)
    e = u - b * z
    se = np.sqrt(np.sum(z ** 2 * e ** 2)) / (z @ z)
    return dict(b=b, t=b / se, F=(b / se) ** 2, F_hom=(b / (np.sqrt(e @ e / (len(z) - 1) / (z @ z)))) ** 2,
                r2=1 - (e @ e) / (u @ u))


def proxy_impact(U, z, scale=1.0):
    """External-instrument identification (Stock and Watson 2012; Mertens and Ravn 2013): the impact column is
    proportional to Cov(u, z); normalised so that the first variable moves by `scale` on impact."""
    c = (U - U.mean(axis=0)).T @ (z - z.mean())
    return scale * c / c[0]


def proxy_irf(m, z_full, H, scale=1.0):
    """Impulse responses to the instrumented shock; z_full aligned with the VAR sample (NaN outside its span)."""
    z = np.asarray(z_full, float)[m['p']:]
    ok = ~np.isnan(z)
    b = proxy_impact(m['U'][ok], z[ok], scale)
    return ma_coefs(m['A'], H) @ b, first_stage(m['U'][ok], z[ok])


def gk_model():
    r = ramey_monthly()
    d = r[GK['vars']].loc[GK['start']:GK['end']].copy()
    for c in ('lip', 'lcpi'):
        d[c] = 100 * d[c]
    z = r['ff4_tc'].loc[GK['start']:GK['end']].copy()
    z[z.index < GK['zstart']] = np.nan
    return d, z, var_ols(d.values, GK['p'])


def lp_iv_se(y, x, z, W, L):
    """Newey-West standard error (L lags) of the 2SLS coefficient of y on x instrumented by z, controls W."""
    D = np.column_stack([y, x, z, W])
    ok = ~np.isnan(D).any(axis=1)
    b, e, Pz = tsls(y[ok], x[ok], z[ok], W[ok])
    return float(np.sqrt(nw_se(Pz, e, L)[0, 0]))


def fig_lp_gk(save_it=True):
    """LP-IV with the same instrument (Ramey 2016, Figure 3B specification: 2 lags, 1990:1-2012:6) against the proxy
    SVAR: the 1-year yield is the endogenous regressor, FF4 its instrument."""
    r = ramey_monthly()
    d = r[['gs1', 'lip', 'lcpi', 'ebp', 'ff4_tc']].copy()
    d['lip'] *= 100
    d['lcpi'] *= 100
    H = LPGK['H']
    # origins t in 1990:1-2012:6; the leads y_{t+h} may reach beyond 2012:6
    full = d.loc[:'2015-12-01']
    Wf = lags_of(full, ['gs1', 'lip', 'lcpi', 'ebp', 'ff4_tc'], LPGK['p'])
    t0 = full.index.get_loc(pd.Timestamp(LPGK['start']))
    t1 = full.index.get_loc(pd.Timestamp(LPGK['end'])) + 1
    res = {}
    for v in ['gs1', 'lip', 'lcpi', 'ebp']:
        bs, ses, Fs = [], [], []
        for h in range(H + 1):
            y = full[v].shift(-h).values[t0:t1]
            x, z, W = full['gs1'].values[t0:t1], full['ff4_tc'].values[t0:t1], Wf[t0:t1]
            b, se, F = lp_iv(y, x, z, W, 0)
            bs.append(b[0])
            ses.append(lp_iv_se(y, x, z, W, h + 1))
            Fs.append(F[0])
        res[v] = (0.25 * np.array(bs), 0.25 * np.array(ses), np.array(Fs))
    _, zg, mg = gk_model()
    th, _ = proxy_irf(mg, zg.values, H, scale=0.25)
    titles = ['1-year Treasury yield (pp)', 'Industrial production (%)', 'CPI (%)', 'Excess bond premium (pp)']
    fig, axs = plt.subplots(1, 4, figsize=(13, 3.4))
    hh = np.arange(H + 1)
    for i, (ax, v) in enumerate(zip(axs, ['gs1', 'lip', 'lcpi', 'ebp'])):
        b, se, _ = res[v]
        band_plot(ax, hh, b, b - 1.645 * se, b + 1.645 * se, st.IDAred, 'LP-IV')
        ax.plot(hh, th[:, i], color=st.MainBlue, lw=1.6, ls='--', label='proxy SVAR')
        ax.set_title(titles[i])
        ax.set_xlabel('months')
    st.fig_legend_bottom(fig, [axs[0].lines[0], patch(st.IDAred), axs[0].lines[2]],
                         ['LP-IV response', '90% band (Newey-West)', 'proxy SVAR (same instrument)'], ncol=3, y=0.0)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save('ats_ch3_lp_gk', save_it)
    ip, cpi = res['lip'], res['lcpi']
    return dict(F0=float(res['gs1'][2][0]), ip24=float(ip[0][24]), ip24_se=float(ip[1][24]), cpi24=float(cpi[0][24]),
                cpi24_se=float(cpi[1][24]), svar_ip24=float(th[24, 1]), svar_cpi24=float(th[24, 2]),
                ratio_se_ip24=float(ip[1][24] / max(1e-9, abs(ip[0][24]))),
                ebp0=float(res['ebp'][0][0]), ebp0_se=float(res['ebp'][1][0]), T=int(t1 - t0))


def sim_dgp(T, rng, burn=100):
    """x_t = 0.5 x_{t-1} + e_t; y_t = 0.6 y_{t-1} + sum_j PSI_j e_{t-j} + u_t: a finite-order VAR is misspecified."""
    n = T + burn
    e, u = rng.standard_normal(n), rng.standard_normal(n)
    x, y = np.zeros(n), np.zeros(n)
    L = len(PSI)
    for t in range(1, n):
        x[t] = 0.5 * x[t - 1] + e[t]
        y[t] = 0.6 * y[t - 1] + sum(PSI[j] * e[t - j] for j in range(min(L, t + 1))) + u[t]
    return np.column_stack([x, y])[burn:]


def true_irf(H):
    """Response of y to a unit shock e (x ordered first): y_h = 0.6 y_{h-1} + PSI_h."""
    r = np.zeros(H + 1)
    for h in range(H + 1):
        r[h] = (0.6 * r[h - 1] if h else 0) + (PSI[h] if h < len(PSI) else 0)
    return r


def lp_sim_estimates(Y, p, H):
    """Recursive LP: y_{t+h} on x_t, controls p lags of (x, y) (x_t is the shock up to scale, ordered first)."""
    x, y = Y[:, 0], Y[:, 1]
    W = np.column_stack([np.r_[np.full(j, np.nan), Y[:-j, k]] for k in range(2) for j in range(1, p + 1)])
    b, _ = lp(y, x, W, H)
    sx = np.nanstd(x[p:] - np.column_stack([np.ones(len(x) - p), W[p:]]) @ np.linalg.lstsq(
        np.column_stack([np.ones(len(x) - p), W[p:]]), x[p:], rcond=None)[0])
    return b * sx                                              # response to a one-s.d. shock


def fig_lp_sim(save_it=True, reps=500, T=240, H=20):
    """Bias and standard deviation of VAR(2), VAR(12) and LP(2) estimates of the impulse response (Li,
    Plagborg-Moller and Wolf 2024): the misspecified short VAR is biased but precise; LP is nearly unbiased but noisy."""
    rng = np.random.default_rng(SEED)
    tr = true_irf(H)
    est = {'VAR(2)': [], 'VAR(12)': [], 'LP(2)': []}
    for _ in range(reps):
        Y = sim_dgp(T, rng)
        for name, p in (('VAR(2)', 2), ('VAR(12)', 12)):
            m = var_ols(Y, p)
            est[name].append(irf(m['A'], np.linalg.cholesky(m['Sigma']), H)[:, 1, 0])
        est['LP(2)'].append(lp_sim_estimates(Y, 2, H))
    fig, axs = plt.subplots(1, 3, figsize=(12.5, 3.5))
    hh = np.arange(H + 1)
    cols = {'VAR(2)': st.MainBlue, 'VAR(12)': st.Forest, 'LP(2)': st.IDAred}
    out = {}
    for k, v in est.items():
        v = np.array(v)
        bias, sd = v.mean(axis=0) - tr, v.std(axis=0)
        out[k] = dict(bias=bias.tolist(), sd=sd.tolist(), rmse=np.sqrt(bias ** 2 + sd ** 2).tolist())
        axs[0].plot(hh, v.mean(axis=0), color=cols[k], lw=1.6, label=k)
        axs[1].plot(hh, np.abs(bias), color=cols[k], lw=1.6, label=k)
        axs[2].plot(hh, sd, color=cols[k], lw=1.6, label=k)
    axs[0].plot(hh, tr, color=st.DarkText, lw=1.2, ls='--', label='true response')
    for ax, t in zip(axs, ['Mean estimate and true response', 'Absolute bias', 'Standard deviation']):
        ax.set_title(t)
        ax.set_xlabel('horizon')
    st.fig_legend_bottom(fig, ncol=4, y=0.0)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save('ats_ch3_lp_sim', save_it)
    def at(k, s, h):
        return out[k][s][h]
    return dict(reps=reps, T=T, peak=float(tr.max()), argpeak=int(tr.argmax()),
                **{f'{k}_{s}_{h}': at(k, s, h) for k in ('VAR(2)', 'VAR(12)', 'LP(2)') for s in ('bias', 'sd', 'rmse')
                   for h in (4, 8, 12, 16)}, true8=float(tr[8]))

## Run

In [ ]:
print(fig_lp_gk())
print(fig_lp_sim(reps=200))

![ats_ch3_lp_gk](./ats_ch3_lp_gk.png)

Output: `ats_ch3_lp_gk.pdf`

![ats_ch3_lp_sim](./ats_ch3_lp_sim.png)

Output: `ats_ch3_lp_sim.pdf`